In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1 Anomaly Detection Benchmark (`models/benchmark_layer1_anomaly_detectors.ipynb`)

This notebook evaluates **Layer 1 ESI 1 Anomaly Detection** using **ONLY 7 RAW TRIAGE INPUT FEATURES** (no feature-engineered inputs):
1. `age`
2. `cc_breathingdifficulty`
3. `gender`
4. `triage_vital_hr`
5. `triage_vital_sbp`
6. `triage_vital_rr`
7. `triage_vital_o2`

### Tested Anomaly Detection Algorithms (Trained Strictly on Normal Non-ESI 1 Rows Without Resampling):
1. **One-Class SVM (`OneClassSVM`)**: Fits a non-linear decision boundary around normal triage vitals.
2. **Isolation Forest (`IsolationForest`)**: Measures path lengths in random isolation trees to detect sparse ESI 1 vitals.
3. **K-Means Centroid Distance Detector (`KMeans`)**: Fits $K=20$ reference centroids on normal vitals and measures Euclidean distance $d_{\min}(x)$ to the nearest cluster center.
4. **Baseline LightGBM Classifier**: Standard supervised binary classifier trained without resampling.

### Evaluated Metrics on Holdout Test Set:
- **ESI 1 Recall (Sensitivity)**
- **Non-ESI 1 Specificity**
- **Balanced Accuracy**
- **ROC-AUC Score**
- **Confusion Matrices** exported to `reports/layer1_anomaly_benchmark_report.csv` & `plots/layer1_anomaly_metrics_barchart.png`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Construct 7 Raw Input Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(pROC)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
# Only 7 Raw Input Features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
train_py <<- train_df
val_py   <<- val_df
test_py  <<- test_df
cat(sprintf("7 Raw Inputs Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Fit Anomaly Detectors on 7 Raw Features in Python
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, roc_auc_score
import lightgbm as lgb
# Retrieve data from R
try:
    pandas2ri.activate()
    train_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_py']))
    test_df  = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['test_py']))
except Exception:
    train_df = pd.DataFrame(r['train_py'])
    test_df  = pd.DataFrame(r['test_py'])
raw_7_cols  = ['age', 'gender', 'cc_breathingdifficulty', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2']
binary_cols = ['gender', 'cc_breathingdifficulty']
cont_cols   = [c for c in raw_7_cols if c not in binary_cols]
# Scale continuous features among 7 raw inputs
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[cont_cols])
X_test_cont  = scaler.transform(test_df[cont_cols])
X_train = np.hstack([X_train_cont, train_df[binary_cols].values])
X_test  = np.hstack([X_test_cont,  test_df[binary_cols].values])
y_train_esi1 = (train_df['target_col'].astype(str).values == '1').astype(int)
y_test_esi1  = (test_df['target_col'].astype(str).values == '1').astype(int)
# Filter NORMAL Non-ESI 1 training data
normal_idx = np.where(y_train_esi1 == 0)[0]
X_train_normal = X_train[normal_idx]
print(f"Training Anomaly Detectors on 7 Raw Features across {len(X_train_normal)} NORMAL Non-ESI 1 samples...")
# ---------------------------------------------------------
# 1. One-Class SVM (Trained on Representative Subsample for Fast Execution)
# ---------------------------------------------------------
np.random.seed(42)
sub_sample_idx = np.random.choice(len(X_train_normal), size=20000, replace=False)
print("Training One-Class SVM (nu=0.01, RBF kernel on 7 Raw Features)...")
oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.01)
oc_svm.fit(X_train_normal[sub_sample_idx])
scores_svm = -oc_svm.decision_function(X_test)
# ---------------------------------------------------------
# 2. Isolation Forest
# ---------------------------------------------------------
print("Training Isolation Forest (n_estimators=100, contamination=0.01 on 7 Raw Features)...")
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_normal)
scores_if = -iso_forest.score_samples(X_test)
# ---------------------------------------------------------
# 3. K-Means Centroid Distance Detector
# ---------------------------------------------------------
print("Training K-Means (K=20 Centroids on 7 Raw Features)...")
kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
kmeans.fit(X_train_normal)
dist_mat = kmeans.transform(X_test)
scores_km = np.min(dist_mat, axis=1)
# ---------------------------------------------------------
# 4. Baseline Supervised LightGBM (Unsampled Binary on 7 Raw Features)
# ---------------------------------------------------------
print("Training Baseline Supervised LightGBM (Unsampled Binary on 7 Raw Features)...")
lgb_base = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, max_depth=6, random_state=42, verbose=-1)
lgb_base.fit(X_train, y_train_esi1)
scores_lgb = lgb_base.predict_proba(X_test)[:, 1]
print("Model training complete on 7 raw features!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Compute & Compare Benchmark Metrics on Holdout Test Set
# ---------------------------------------------------------
def evaluate_anomaly_detector(name, scores, y_true):
    auc = roc_auc_score(y_true, scores)
    
    # Determine optimal decision threshold via percentile matching target ESI 1 ratio (~1%)
    threshold = np.percentile(scores, 99.0)
    y_pred = (scores >= threshold).astype(int)
    
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    tp = cm[0, 0]
    fn = cm[0, 1]
    fp = cm[1, 0]
    tn = cm[1, 1]
    
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    bal  = (rec + spec) / 2.0
    
    print(f"============================================================")
    print(f"   LAYER 1 ESI 1 (7 RAW INPUTS): {name.upper()}")
    print(f"============================================================")
    print(f"  ESI 1 Sensitivity (Recall) : {rec:.4f}")
    print(f"  Non-ESI 1 Specificity      : {spec:.4f}")
    print(f"  Balanced Accuracy          : {bal:.4f}")
    print(f"  ROC-AUC Score              : {auc:.4f}")
    print(f"============================================================\n")
    print("Confusion Matrix (Reference = Ground Truth, Prediction = Model):")
    print(cm)
    print("\n")
    
    return {
        'Algorithm': name,
        'Recall_Sensitivity': round(rec, 4),
        'Specificity': round(spec, 4),
        'Balanced_Accuracy': round(bal, 4),
        'ROC_AUC': round(auc, 4)
    }
res_svm = evaluate_anomaly_detector("OneClass_SVM", scores_svm, y_test_esi1)
res_if  = evaluate_anomaly_detector("Isolation_Forest", scores_if, y_test_esi1)
res_km  = evaluate_anomaly_detector("KMeans_Distance", scores_km, y_test_esi1)
res_lgb = evaluate_anomaly_detector("Baseline_LightGBM", scores_lgb, y_test_esi1)
results_df = pd.DataFrame([res_svm, res_if, res_km, res_lgb])
reports_dir = "../reports"
if not os.path.exists(reports_dir): reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, "layer1_anomaly_benchmark_report.csv")
results_df.to_csv(report_path, index=False)
print(f"Benchmark Report saved to: {report_path}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Grouped Bar Chart Visualization
# ---------------------------------------------------------
plots_dir = "../plots"
if not os.path.exists(plots_dir): plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)
plt.figure(figsize=(10, 5.5), dpi=300)
algs = results_df['Algorithm'].unique()
metrics = ['Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'ROC_AUC']
x = np.arange(len(metrics))
width = 0.2
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, alg in enumerate(algs):
    scores = [results_df[results_df['Algorithm'] == alg][m].values[0] for m in metrics]
    plt.bar(x + i * width, scores, width, label=alg, color=colors[i % len(colors)])
    for j, sc in enumerate(scores):
        plt.text(x[j] + i * width, sc + 0.01, f"{sc:.3f}", ha='center', va='bottom', fontsize=8)
plt.title('Layer 1 ESI 1 Anomaly Detector Benchmark (7 Raw Features)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Metric', fontsize=11)
plt.ylabel('Score', fontsize=11)
plt.xticks(x + width * 1.5, ['Recall (Sens)', 'Specificity', 'Bal Accuracy', 'ROC-AUC'])
plt.ylim(0, 1.15)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5, axis='y')
plt.tight_layout()
barchart_path = os.path.join(plots_dir, "layer1_anomaly_metrics_barchart.png")
plt.savefig(barchart_path)
plt.close()
print(f"Bar Chart saved to: {barchart_path}")